In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive_output, FloatSlider, HBox, VBox, Layout, HTML, Output
from IPython.display import display

# ------------------------------------------------------------
# 1. DESCRIPTION
# ------------------------------------------------------------

description = HTML("""
<div style="
    border:1px solid #b9d7f5;
    border-radius:8px;
    padding:8px 10px;
    margin-bottom:10px;
    font-size:13px;
    line-height:1.35;
    background-color:#f7fbff;
">
<div><b>Purpose:</b> Explore the magnitude and phase responses of a second-order series RLC symmetrical band-stop filter as the circuit parameters R, L, and C are varied.</div>
<div><b>What we see:</b> The gain G(ω), the phase response, the notch frequency ω₀, the lower and upper cutoff frequencies ωc1 and ωc2, and the bandwidth Δω.</div>
<div><b>What happens as we interact:</b> Changing R, L, and C modifies ω₀, Q, the cutoff frequencies, and the width of the rejected frequency band.</div>
</div>
""")

# ------------------------------------------------------------
# 2. CONTROLS
# ------------------------------------------------------------

r_slider = FloatSlider(min=1.0, max=100.0, step=1.0, value=20.0, description='R:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'30px'}, layout=Layout(width='220px'))

l_slider = FloatSlider(min=1.0, max=100.0, step=1.0, value=50.0, description='L:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'30px'}, layout=Layout(width='220px'))

c_slider = FloatSlider(min=1.0, max=200.0, step=1.0, value=100.0, description='C:', continuous_update=True, readout=True, readout_format='.1f', style={'description_width':'30px'}, layout=Layout(width='220px'))

# ------------------------------------------------------------
# 3. LEGEND
# ------------------------------------------------------------

legend_html = HTML("""
<div style="
    border:1px solid #cccccc;
    border-radius:5px;
    padding:7px 9px;
    width:135px;
    font-size:13px;
    line-height:1.7;
    background:white;
">
<div><span style="display:inline-block; width:32px; border-top:3px solid red; vertical-align:middle; margin-right:7px;"></span>G(ω)</div>
<div><span style="display:inline-block; width:32px; border-top:2px dashed #888888; vertical-align:middle; margin-right:7px;"></span>1/√2</div>
<div><span style="display:inline-block; width:32px; border-top:2px dashed black; vertical-align:middle; margin-right:7px;"></span>ωc1</div>
<div><span style="display:inline-block; width:32px; border-top:2px dotted black; vertical-align:middle; margin-right:7px;"></span>ω₀</div>
<div><span style="display:inline-block; width:32px; border-top:2px dash-dot #888888; vertical-align:middle; margin-right:7px;"></span>ωc2</div>
</div>
""")

parameter_label = HTML("<div style='font-size:14px; font-weight:bold; margin-top:12px; margin-bottom:4px;'>Circuit Parameters:</div>")

units_html = HTML("""
<div style="
    font-size:12px;
    line-height:1.5;
    margin-top:7px;
    color:#555555;
">
R in Ω<br>
L in mH<br>
C in μF
</div>
""")

# ------------------------------------------------------------
# 4. SEPARATE OUTPUT AREAS
# ------------------------------------------------------------

magnitude_output = Output(layout=Layout(width='100%', overflow='hidden'))

phase_output = Output(layout=Layout(width='100%', overflow='hidden'))

info_output = Output(layout=Layout(width='100%', overflow='hidden'))

# ------------------------------------------------------------
# 5. MAIN FUNCTION
# ------------------------------------------------------------

def update_rlc_bandstop(R, L, C):

    L_si = L * 1e-3
    C_si = C * 1e-6

    # --------------------------------------------------------
    # FILTER CHARACTERISTICS
    # --------------------------------------------------------

    omega0 = 1.0 / np.sqrt(L_si * C_si)

    Q = np.sqrt(L_si / (C_si * R**2))

    omega_c1 = -R / (2.0 * L_si) + np.sqrt((R / (2.0 * L_si))**2 + 1.0 / (L_si * C_si))

    omega_c2 = R / (2.0 * L_si) + np.sqrt((R / (2.0 * L_si))**2 + 1.0 / (L_si * C_si))

    bandwidth = omega_c2 - omega_c1

    # --------------------------------------------------------
    # FREQUENCY AXIS
    # --------------------------------------------------------

    omega_axis_max = max(2.5 * omega0, 2.2 * omega_c2)

    omega = np.linspace(0.001, omega_axis_max, 4000)

    # --------------------------------------------------------
    # FREQUENCY RESPONSE
    #
    # H(jω) = (ω₀² - ω²) /
    #         [(ω₀² - ω²) + j(R/L)ω]
    # --------------------------------------------------------

    numerator = omega0**2 - omega**2

    denominator = (omega0**2 - omega**2) + 1j * (R / L_si) * omega

    H = numerator / denominator

    G = np.abs(H)

    phase = np.angle(H, deg=True)

    cutoff_gain = 1.0 / np.sqrt(2.0)

    # --------------------------------------------------------
    # 6. MAGNITUDE RESPONSE
    # --------------------------------------------------------

    with magnitude_output:

        magnitude_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.5))

        ax.plot(omega, G, 'r-', linewidth=2.0)

        ax.axhline(cutoff_gain, color='gray', linestyle='--', linewidth=1.2)

        ax.axvline(omega_c1, color='black', linestyle='--', linewidth=1.5)

        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        ax.axvline(omega_c2, color='gray', linestyle='-.', linewidth=1.5)

        ax.plot(omega_c1, cutoff_gain, 'ko', markersize=4)

        ax.plot(omega0, 0.0, 'ko', markersize=4)

        ax.plot(omega_c2, cutoff_gain, 'ko', markersize=4)

        arrow_y = 0.12

        ax.annotate('', xy=(omega_c2, arrow_y), xytext=(omega_c1, arrow_y), arrowprops=dict(arrowstyle='<->', linewidth=1.2))

        ax.text((omega_c1 + omega_c2) / 2.0, arrow_y + 0.06, 'Δω', horizontalalignment='center', fontsize=10)

        ax.set_xlim(0.0, omega_axis_max)

        ax.set_ylim(0.0, 1.2)

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)

        ax.set_ylabel('Gain G(ω)', fontsize=11)

        ax.set_title('Magnitude Response of the Series RLC Band-Stop Filter', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, linestyle=':', alpha=0.35)

        plt.tight_layout()

        plt.show()

        plt.close(fig)

    # --------------------------------------------------------
    # 7. PHASE RESPONSE
    # --------------------------------------------------------

    with phase_output:

        phase_output.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8.2, 3.2))

        ax.plot(omega, phase, 'r-', linewidth=2.0)

        ax.axhline(0.0, color='gray', linestyle='--', linewidth=1.0)

        ax.axvline(omega_c1, color='black', linestyle='--', linewidth=1.5)

        ax.axvline(omega0, color='black', linestyle=':', linewidth=1.5)

        ax.axvline(omega_c2, color='gray', linestyle='-.', linewidth=1.5)

        ax.set_xlim(0.0, omega_axis_max)

        ax.set_ylim(-100.0, 100.0)

        ax.set_yticks([-90, -45, 0, 45, 90])

        ax.set_xlabel('Angular Frequency ω (rad/s)', fontsize=11)

        ax.set_ylabel('Phase (degrees)', fontsize=11)

        ax.set_title('Phase Response of the Series RLC Band-Stop Filter', fontsize=13, fontweight='bold', pad=8)

        ax.grid(True, linestyle=':', alpha=0.35)

        plt.tight_layout()

        plt.show()

        plt.close(fig)

    # --------------------------------------------------------
    # 8. CHECKS
    # --------------------------------------------------------

    product_check = omega_c1 * omega_c2

    omega0_squared = omega0**2

    bandwidth_check = R / L_si

    # --------------------------------------------------------
    # 9. INFORMATION FRAME
    # --------------------------------------------------------

    info_html = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:7px 11px;
        font-size:12.5px;
        background:white;
        width:fit-content;
        max-width:100%;
        box-sizing:border-box;
    ">

        <div style="
            display:flex;
            flex-direction:row;
            align-items:center;
            gap:18px;
            white-space:nowrap;
        ">
            <div><b>R:</b> <span style="color:#0066cc;">{R:.1f} Ω</span></div>
            <div><b>L:</b> <span style="color:#0066cc;">{L:.1f} mH</span></div>
            <div><b>C:</b> <span style="color:#0066cc;">{C:.1f} μF</span></div>
            <div><b>ω₀:</b> <span style="color:#0066cc;">{omega0:.2f} rad/s</span></div>
            <div><b>Q:</b> <span style="color:#0066cc;">{Q:.3f}</span></div>
        </div>

        <div style="
            margin-top:5px;
            padding-top:5px;
            border-top:1px solid #eeeeee;
            display:flex;
            flex-direction:row;
            align-items:center;
            gap:18px;
            white-space:nowrap;
        ">
            <div><b>ωc1:</b> <span style="color:#0066cc;">{omega_c1:.2f} rad/s</span></div>
            <div><b>ωc2:</b> <span style="color:#0066cc;">{omega_c2:.2f} rad/s</span></div>
            <div><b>Δω:</b> <span style="color:#0066cc;">{bandwidth:.2f} rad/s</span></div>
            <div><b>G(ω₀):</b> <span style="color:#0066cc;">0</span></div>
        </div>

        <div style="
            margin-top:5px;
            padding-top:5px;
            border-top:1px solid #eeeeee;
            white-space:nowrap;
        ">
            <b>Checks:</b>
            <span style="margin-left:12px;">ωc1 × ωc2 = <span style="color:#0066cc;">{product_check:.2f}</span></span>
            <span style="margin-left:18px;">ω₀² = <span style="color:#0066cc;">{omega0_squared:.2f}</span></span>
            <span style="margin-left:18px;">R/L = <span style="color:#0066cc;">{bandwidth_check:.2f} rad/s</span></span>
        </div>

    </div>
    """

    with info_output:

        info_output.clear_output(wait=True)

        display(HTML(info_html))

# ------------------------------------------------------------
# 10. INTERACTION
# ------------------------------------------------------------

interactive_controls = interactive_output(update_rlc_bandstop, {'R': r_slider, 'L': l_slider, 'C': c_slider})

interactive_controls.layout.display = 'none'

# ------------------------------------------------------------
# 11. LEFT COLUMN
# ------------------------------------------------------------

control_column = VBox([legend_html, parameter_label, r_slider, l_slider, c_slider, units_html], layout=Layout(width='230px', min_width='230px', flex='0 0 230px', align_items='flex-start', padding='0px 0px 0px 4px', overflow='hidden'))

# ------------------------------------------------------------
# 12. RIGHT COLUMN
# ------------------------------------------------------------

right_column = VBox([magnitude_output, phase_output, info_output], layout=Layout(width='auto', min_width='0px', flex='1 1 auto', align_items='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 13. COMPLETE MAIN AREA
# ------------------------------------------------------------

main_area = HBox([control_column, right_column], layout=Layout(width='100%', max_width='100%', align_items='flex-start', justify_content='flex-start', overflow='hidden'))

# ------------------------------------------------------------
# 14. FINAL DISPLAY
# ------------------------------------------------------------

display(description)

display(main_area)

display(interactive_controls)